# 02 - Enclosures & Tessellation

Following source `generate_elements.py`:
1. Generate adaptive buffered limit using Gabriel triangulation of building representative points
2. Create enclosures from streets + buffered limit
3. Run enclosed tessellation
4. QA loop: retry if some buildings are missing tessellation cells

Source uses `mm.buffered_limit(buildings, buffer='adaptive')` or a Gabriel-based buffer.
We follow the Gabriel approach (`generate_enclosures_representative_points`).

In [1]:
import os
if os.path.basename(os.getcwd()).startswith("notebooks"):
    os.chdir(os.path.dirname(os.getcwd()))
print("Working dir:", os.getcwd())

import sys
sys.path.insert(0, "scripts")
from pipeline_config import load_city_config
load_city_config()

Working dir: C:\urban-form\urban_taxonomy
[pipeline_config] City config: config\canada.env


WindowsPath('config/canada.env')

In [2]:
import geopandas as gpd
import numpy as np
import momepy as mm
from libpysal.graph import Graph
from shapely import GEOSException
from shapely import Point
import warnings
from pathlib import Path

warnings.filterwarnings("ignore")

OUTPUT_ROOT = os.environ.get("OUTPUT_ROOT", "outputs")
PREPROCESSED_DIR = Path(OUTPUT_ROOT) / "01_preprocessed"
OUT = Path(OUTPUT_ROOT) / "02_tessellation"
OUT.mkdir(parents=True, exist_ok=True)

## 1. Load preprocessed data

In [3]:
buildings = gpd.read_file(PREPROCESSED_DIR / "buildings.gpkg")
roads     = gpd.read_file(PREPROCESSED_DIR / "roads.gpkg")

buildings = buildings.reset_index(drop=True)
buildings.index = buildings.index.astype(int)

# Remove invalid/empty geometries
buildings = buildings[~buildings.geometry.is_empty & buildings.geometry.is_valid].copy()
buildings = buildings.reset_index(drop=True)
print(f"{len(buildings):,} buildings, {len(roads):,} roads")

571,847 buildings, 62,718 roads


## 2. Generate enclosures

Source `generate_enclosures_representative_points()` builds a Gabriel triangulation
of building representative points, takes `max_dist/2 + 10%` as the buffer per building,
clips to [0, 100] m, then unions and uses as the enclosure limit.

In [4]:
# Adaptive buffer via Gabriel triangulation (source approach)
MIN_BUFFER = 0.0
MAX_BUFFER = 100.0

gabriel = Graph.build_triangulation(
    buildings.representative_point(), "gabriel", kernel="identity"
)
max_dist = gabriel.aggregate("max")
buffer = np.clip(max_dist / 2 + max_dist * 0.1, MIN_BUFFER, MAX_BUFFER).values
buffered_buildings = buildings.buffer(buffer, resolution=2).union_all()

enclosures = mm.enclosures(roads, limit=buffered_buildings, clip=True)
print(f"{len(enclosures):,} enclosures generated")

51,694 enclosures generated


## 3. Enclosed tessellation

Source `generate_tess()` with QA loop:
- Run `mm.enclosed_tessellation()`
- Dissolve split cells
- Verify every building has a tessellation cell
- If not, drop problem buildings and retry

In [5]:
import os

# momepy spins up a fresh joblib/loky worker pool (all cores) on every call
# when n_jobs=-1. For a region-chunked run (scripts/region_pipeline.py) most
# regions are small, and repeatedly spinning up/tearing down a 16-process
# pool for a few hundred buildings is both slower than single-threaded and
# prone to a harmless-but-noisy joblib/loky Windows race in the background
# resource_tracker process on teardown. Only pay for parallelism once it's
# actually worth it.
N_WORKERS = -1 if len(buildings) >= 20_000 else 1
print(f"n_jobs for tessellation: {N_WORKERS} ({len(buildings):,} buildings)")


def generate_tess(buildings, enclosures, n_workers=N_WORKERS):
    """Source generate_tess() — run tessellation and clean up."""
    tessellation = mm.enclosed_tessellation(
        buildings, enclosures.geometry, simplify=True, n_jobs=n_workers
    )
    # dissolve split buildings
    tessellation = tessellation.dissolve(by=tessellation.index.values)

    # drop empty spaces with no buildings but positive index
    tessellation = tessellation.explode()
    inp, res = buildings.geometry.centroid.sindex.query(tessellation.geometry)
    to_keep = np.append(np.unique(inp), np.where(tessellation.index.values < 0)[0])
    tessellation = tessellation.iloc[to_keep]

    # drop remaining duplicates
    tessellation = tessellation[~tessellation.index.duplicated()].sort_index()
    return tessellation


# Run with QA loop (source generate_elements.py logic)
try:
    tessellation = generate_tess(buildings, enclosures)
except GEOSException as e:
    txt = str(e)
    print("Topology problem:", txt[:100])
    coords = [float(s) for s in txt[45:].split(". T")[0].split(" ")]
    problem_point = Point(*coords)
    buildings = buildings.drop(buildings.sindex.query(problem_point, predicate="intersects"))
    tessellation = generate_tess(buildings, enclosures)

# QA: ensure every building has a tessellation cell
coverage = np.isin(buildings.index.values, tessellation.index.values)
num_dropped = 0
while not coverage.all():
    print(f"Retrying tessellation: {(~coverage).sum()} buildings missing cells")
    num_dropped += (~coverage).sum()
    buildings = buildings[coverage].reset_index(drop=True)
    enclosures = mm.enclosures(roads, limit=buffered_buildings, clip=True)
    tessellation = generate_tess(buildings, enclosures)
    coverage = np.isin(buildings.index.values, tessellation.index.values)

assert coverage.all(), "Not all buildings have tessellation cells!"

if num_dropped:
    print(f"Dropped {num_dropped} buildings due to tessellation problems")
    buildings.to_file(PREPROCESSED_DIR / "buildings.gpkg", driver="GPKG")

print(f"Tessellation: {len(tessellation):,} cells ({(tessellation.index >= 0).sum():,} with buildings)")

n_jobs for tessellation: -1 (571,847 buildings)


Tessellation: 594,850 cells (571,847 with buildings)


## 4. Save

In [6]:
tessellation.to_parquet(OUT / "tessellation.parquet")
enclosures.to_parquet(OUT / "enclosures.parquet")

print(f"Saved to {OUT}")
print(f"  tessellation.parquet : {len(tessellation):,} cells")
print(f"  enclosures.parquet   : {len(enclosures):,} enclosures")

Saved to outputs_canada\regions\4811\02_tessellation
  tessellation.parquet : 594,850 cells
  enclosures.parquet   : 51,694 enclosures
